In [2]:
from wds_astropy_table import parse_wdsweb
from wds_astropy_table import parse_sixth_orbital

from astropy.table import Table
from astropy.io import ascii

import numpy as np
import sys

table_wds = parse_wdsweb('wdsweb_summ2.txt')
#table_orbital = parse_sixth_orbital('orb6orbits.txt')



In [3]:
discoverer="STF 518"
#wds_id="04153-0739"
components="BC"

normalized=discoverer.replace(" ", "")+components
wds = table_wds[(table_wds['discoverer_normalized']==normalized) & (table_wds['components']==components)][0]
#wds = table_wds[(table_wds['id']==wds_id) & (table_wds['components']==components)][0]
wds

id,discoverer,components,first_date,last_date,num_obs,first_pa,last_pa,first_sep,last_sep,pri_mag,sec_mag,spectral,pri_pm_ra,pri_pm_dec,sec_pm_ra,sec_pm_dec,dm,notes,j2000,discoverer_normalized
str10,str7,str5,int64,int64,int64,int64,int64,float64,float64,float64,float64,str9,float64,float64,float64,float64,str8,str4,str18,str12
04153-0739,STF 518,BC,1851,2024,192,160,327,3.0,7.7,9.53,11.17,DA2.9+M5V,-224.0,-334.0,-225.0,-341.0,-07 781,NO P,041521.79-073929.1,STF518BC


In [4]:
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.time import Time

# "wds coordinate string"
wcs = wds['j2000']

# parse into format recognized by skycoord
formatted_coord_string = wcs[0:2]+'h'+wcs[2:4]+'m'+wcs[4:9]+'s' + ' ' + wcs[9:12]+'d'+wcs[12:14]+'m'+wcs[14:18]+'s'
print(wcs,"=>", formatted_coord_string)

# get the position angle and offset of secondary
pa = wds['last_pa']*u.deg
sep = wds['last_sep']*u.arcsec

# create skycoord objects
c_pri_j2000 = SkyCoord(formatted_coord_string, unit=(u.hourangle, u.deg), obstime=Time('J2000.0'))
c_sec_j2000 = c_pri_j2000.directional_offset_by(pa,sep)
print(c_pri_j2000.separation(c_sec_j2000))
print(c_pri_j2000.position_angle(c_sec_j2000).to(u.deg))

# adjust to gaia epoch
c_pri = c_pri_j2000.transform_to(SkyCoord(ra=0*u.deg, dec=0*u.deg, frame='icrs', equinox='J2016.0'))
c_sec = c_sec_j2000.transform_to(SkyCoord(ra=0*u.deg, dec=0*u.deg, frame='icrs', equinox='J2016.0'))

# Display the coordinates in ICRS (Gaia standard)
print(f"Primary j2000:   {c_pri.to_string('hmsdms')}")
print(f"Secondary j2000: {c_sec.to_string('hmsdms')}")
print(f"Primary j2016:   {c_pri.to_string('hmsdms')}")
print(f"Secondary j2016: {c_sec.to_string('hmsdms')}")


041521.79-073929.1 => 04h15m21.79s -07d39m29.1s
0d00m07.7s
327d00m00.00000021s
Primary j2000:   04h15m21.79s -07d39m29.1s
Secondary j2000: 04h15m21.50790378s -07d39m22.64223089s
Primary j2016:   04h15m21.79s -07d39m29.1s
Secondary j2016: 04h15m21.50790378s -07d39m22.64223089s


In [5]:
# do a Gaia cone search for the primary and secondary
from astroquery.gaia import Gaia

search_radius = 8*u.arcsec

job = Gaia.cone_search_async(c_pri, radius=search_radius)
results_pri = job.get_results()

job = Gaia.cone_search_async(c_sec, radius=search_radius)
results_sec = job.get_results()

display(results_pri)
display(results_sec)
#results_t = Table()
#results_t = vstack([results_t, results])

print(f"Primary source_id={results_pri['source_id'][0]}")
print(f"Secondary source_id={results_sec['source_id'][0]}")

INFO: Query finished. [astroquery.utils.tap.core]
INFO: Query finished. [astroquery.utils.tap.core]


solution_id,designation,source_id,random_index,ref_epoch,ra,ra_error,dec,dec_error,parallax,parallax_error,parallax_over_error,pm,pmra,pmra_error,pmdec,pmdec_error,ra_dec_corr,ra_parallax_corr,ra_pmra_corr,ra_pmdec_corr,dec_parallax_corr,dec_pmra_corr,dec_pmdec_corr,parallax_pmra_corr,parallax_pmdec_corr,pmra_pmdec_corr,astrometric_n_obs_al,astrometric_n_obs_ac,astrometric_n_good_obs_al,astrometric_n_bad_obs_al,astrometric_gof_al,astrometric_chi2_al,astrometric_excess_noise,astrometric_excess_noise_sig,astrometric_params_solved,astrometric_primary_flag,nu_eff_used_in_astrometry,pseudocolour,pseudocolour_error,ra_pseudocolour_corr,dec_pseudocolour_corr,parallax_pseudocolour_corr,pmra_pseudocolour_corr,pmdec_pseudocolour_corr,astrometric_matched_transits,visibility_periods_used,astrometric_sigma5d_max,matched_transits,new_matched_transits,matched_transits_removed,ipd_gof_harmonic_amplitude,ipd_gof_harmonic_phase,ipd_frac_multi_peak,ipd_frac_odd_win,ruwe,scan_direction_strength_k1,scan_direction_strength_k2,scan_direction_strength_k3,scan_direction_strength_k4,scan_direction_mean_k1,scan_direction_mean_k2,scan_direction_mean_k3,scan_direction_mean_k4,duplicated_source,phot_g_n_obs,phot_g_mean_flux,phot_g_mean_flux_error,phot_g_mean_flux_over_error,phot_g_mean_mag,phot_bp_n_obs,phot_bp_mean_flux,phot_bp_mean_flux_error,phot_bp_mean_flux_over_error,phot_bp_mean_mag,phot_rp_n_obs,phot_rp_mean_flux,phot_rp_mean_flux_error,phot_rp_mean_flux_over_error,phot_rp_mean_mag,phot_bp_rp_excess_factor,phot_bp_n_contaminated_transits,phot_bp_n_blended_transits,phot_rp_n_contaminated_transits,phot_rp_n_blended_transits,phot_proc_mode,bp_rp,bp_g,g_rp,radial_velocity,radial_velocity_error,rv_method_used,rv_nb_transits,rv_nb_deblended_transits,rv_visibility_periods_used,rv_expected_sig_to_noise,rv_renormalised_gof,rv_chisq_pvalue,rv_time_duration,rv_amplitude_robust,rv_template_teff,rv_template_logg,rv_template_fe_h,rv_atm_param_origin,vbroad,vbroad_error,vbroad_nb_transits,grvs_mag,grvs_mag_error,grvs_mag_nb_transits,rvs_spec_sig_to_noise,phot_variable_flag,l,b,ecl_lon,ecl_lat,in_qso_candidates,in_galaxy_candidates,non_single_star,has_xp_continuous,has_xp_sampled,has_rvs,has_epoch_photometry,has_epoch_rv,has_mcmc_gspphot,has_mcmc_msc,in_andromeda_survey,classprob_dsc_combmod_quasar,classprob_dsc_combmod_galaxy,classprob_dsc_combmod_star,teff_gspphot,teff_gspphot_lower,teff_gspphot_upper,logg_gspphot,logg_gspphot_lower,logg_gspphot_upper,mh_gspphot,mh_gspphot_lower,mh_gspphot_upper,distance_gspphot,distance_gspphot_lower,distance_gspphot_upper,azero_gspphot,azero_gspphot_lower,azero_gspphot_upper,ag_gspphot,ag_gspphot_lower,ag_gspphot_upper,ebpminrp_gspphot,ebpminrp_gspphot_lower,ebpminrp_gspphot_upper,libname_gspphot,dist
,,,,yr,deg,mas,deg,mas,mas,mas,,mas / yr,mas / yr,mas / yr,mas / yr,mas / yr,,,,,,,,,,,,,,,,,mas,,,,1 / um,1 / um,1 / um,,,,,,,,mas,,,,,deg,,,,,,,,deg,deg,deg,deg,,,electron / s,electron / s,,mag,,electron / s,electron / s,,mag,,electron / s,electron / s,,mag,,,,,,,mag,mag,mag,km / s,km / s,,,,,,,,d,km / s,K,log(cm.s**-2),dex,,km / s,km / s,,mag,mag,,,,deg,deg,deg,deg,,,,,,,,,,,,,,,K,K,K,log(cm.s**-2),log(cm.s**-2),log(cm.s**-2),dex,dex,dex,pc,pc,pc,mag,mag,mag,mag,mag,mag,mag,mag,mag,,
int64,object,int64,int64,float64,float64,float32,float64,float32,float64,float32,float32,float32,float64,float32,float64,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,int16,int16,int16,int16,float32,float32,float32,float32,int16,bool,float32,float32,float32,float32,float32,float32,float32,float32,int16,int16,float32,int16,int16,int16,float32,float32,int16,int16,float32,float32,float32,float32,float32,float32,float32,float32,float32,bool,int16,float64,float32,float32,float32,int16,float64,float32,float32,float32,int16,float64,float32,float32,float32,float32,int16,int16,int16,int16,int16,float32,float32,float32,float32,float32,int16,int16,int16,int16,float32,float32,float32,float32,float32,float32,float32,fl

solution_id,designation,source_id,random_index,ref_epoch,ra,ra_error,dec,dec_error,parallax,parallax_error,parallax_over_error,pm,pmra,pmra_error,pmdec,pmdec_error,ra_dec_corr,ra_parallax_corr,ra_pmra_corr,ra_pmdec_corr,dec_parallax_corr,dec_pmra_corr,dec_pmdec_corr,parallax_pmra_corr,parallax_pmdec_corr,pmra_pmdec_corr,astrometric_n_obs_al,astrometric_n_obs_ac,astrometric_n_good_obs_al,astrometric_n_bad_obs_al,astrometric_gof_al,astrometric_chi2_al,astrometric_excess_noise,astrometric_excess_noise_sig,astrometric_params_solved,astrometric_primary_flag,nu_eff_used_in_astrometry,pseudocolour,pseudocolour_error,ra_pseudocolour_corr,dec_pseudocolour_corr,parallax_pseudocolour_corr,pmra_pseudocolour_corr,pmdec_pseudocolour_corr,astrometric_matched_transits,visibility_periods_used,astrometric_sigma5d_max,matched_transits,new_matched_transits,matched_transits_removed,ipd_gof_harmonic_amplitude,ipd_gof_harmonic_phase,ipd_frac_multi_peak,ipd_frac_odd_win,ruwe,scan_direction_strength_k1,scan_direction_strength_k2,scan_direction_strength_k3,scan_direction_strength_k4,scan_direction_mean_k1,scan_direction_mean_k2,scan_direction_mean_k3,scan_direction_mean_k4,duplicated_source,phot_g_n_obs,phot_g_mean_flux,phot_g_mean_flux_error,phot_g_mean_flux_over_error,phot_g_mean_mag,phot_bp_n_obs,phot_bp_mean_flux,phot_bp_mean_flux_error,phot_bp_mean_flux_over_error,phot_bp_mean_mag,phot_rp_n_obs,phot_rp_mean_flux,phot_rp_mean_flux_error,phot_rp_mean_flux_over_error,phot_rp_mean_mag,phot_bp_rp_excess_factor,phot_bp_n_contaminated_transits,phot_bp_n_blended_transits,phot_rp_n_contaminated_transits,phot_rp_n_blended_transits,phot_proc_mode,bp_rp,bp_g,g_rp,radial_velocity,radial_velocity_error,rv_method_used,rv_nb_transits,rv_nb_deblended_transits,rv_visibility_periods_used,rv_expected_sig_to_noise,rv_renormalised_gof,rv_chisq_pvalue,rv_time_duration,rv_amplitude_robust,rv_template_teff,rv_template_logg,rv_template_fe_h,rv_atm_param_origin,vbroad,vbroad_error,vbroad_nb_transits,grvs_mag,grvs_mag_error,grvs_mag_nb_transits,rvs_spec_sig_to_noise,phot_variable_flag,l,b,ecl_lon,ecl_lat,in_qso_candidates,in_galaxy_candidates,non_single_star,has_xp_continuous,has_xp_sampled,has_rvs,has_epoch_photometry,has_epoch_rv,has_mcmc_gspphot,has_mcmc_msc,in_andromeda_survey,classprob_dsc_combmod_quasar,classprob_dsc_combmod_galaxy,classprob_dsc_combmod_star,teff_gspphot,teff_gspphot_lower,teff_gspphot_upper,logg_gspphot,logg_gspphot_lower,logg_gspphot_upper,mh_gspphot,mh_gspphot_lower,mh_gspphot_upper,distance_gspphot,distance_gspphot_lower,distance_gspphot_upper,azero_gspphot,azero_gspphot_lower,azero_gspphot_upper,ag_gspphot,ag_gspphot_lower,ag_gspphot_upper,ebpminrp_gspphot,ebpminrp_gspphot_lower,ebpminrp_gspphot_upper,libname_gspphot,dist
,,,,yr,deg,mas,deg,mas,mas,mas,,mas / yr,mas / yr,mas / yr,mas / yr,mas / yr,,,,,,,,,,,,,,,,,mas,,,,1 / um,1 / um,1 / um,,,,,,,,mas,,,,,deg,,,,,,,,deg,deg,deg,deg,,,electron / s,electron / s,,mag,,electron / s,electron / s,,mag,,electron / s,electron / s,,mag,,,,,,,mag,mag,mag,km / s,km / s,,,,,,,,d,km / s,K,log(cm.s**-2),dex,,km / s,km / s,,mag,mag,,,,deg,deg,deg,deg,,,,,,,,,,,,,,,K,K,K,log(cm.s**-2),log(cm.s**-2),log(cm.s**-2),dex,dex,dex,pc,pc,pc,mag,mag,mag,mag,mag,mag,mag,mag,mag,,
int64,object,int64,int64,float64,float64,float32,float64,float32,float64,float32,float32,float32,float64,float32,float64,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,int16,int16,int16,int16,float32,float32,float32,float32,int16,bool,float32,float32,float32,float32,float32,float32,float32,float32,int16,int16,float32,int16,int16,int16,float32,float32,int16,int16,float32,float32,float32,float32,float32,float32,float32,float32,float32,bool,int16,float64,float32,float32,float32,int16,float64,float32,float32,float32,int16,float64,float32,float32,float32,float32,int16,int16,int16,int16,int16,float32,float32,float32,float32,float32,int16,int16,int16,int16,float32,float32,float32,float32,float32,float32,float32,fl

IndexError: index 0 is out of bounds for axis 0 with size 0

In [15]:
# do a Gaia cone search for the primary and secondary
from astroquery.simbad import Simbad
import re

# extract the Gaia DR3 id from the Simbad ids string
def get_gaia_dr3_id(ids):
    regex = 'GAIA DR3 (\\d+)'
    result = re.search(regex, ids, re.IGNORECASE)

    return result.group(1) 


search_radius = 3*u.arcsec

Simbad.add_votable_fields('ids', 'velocity')#, 'typed_id')
results_pri = Simbad.query_region(c_pri_j2000, radius=search_radius)
#results_pri.sort('dist')
results_sec = Simbad.query_region(c_sec_j2000, radius=search_radius)

gaia_id_pri=get_gaia_dr3_id(results_pri['ids'][0])
gaia_id_sec=get_gaia_dr3_id(results_sec['ids'][0])
print(f"gaia_id_pri={gaia_id_pri}")
print(f"gaia_id_sec={gaia_id_sec}")

display(results_pri)
display(results_sec)
#results_t = Table()
#results_t = vstack([results_t, results])

print(c_pri_j2000.to_string())




gaia_id_pri=3195919254111315712
gaia_id_sec=3195919254111314816


main_id,ra,dec,coo_err_maj,coo_err_min,coo_err_angle,coo_wavelength,coo_bibcode,ids,mesdistance.bibcode,mesdistance.dist,mesdistance.dist_prec,mesdistance.mespos,mesdistance.method,mesdistance.minus_err,mesdistance.minus_err_prec,mesdistance.plus_err,mesdistance.plus_err_prec,mesdistance.qual,mesdistance.unit,rvz_bibcode,rvz_err,rvz_qual,rvz_nature,rvz_err_prec,rvz_radvel,rvz_wavelength,rvz_radvel_prec,rvz_redshift,rvz_redshift_prec,rvz_type
,deg,deg,mas,mas,deg,,,,,,,,,,,,,,,,km / s,,,,km / s,,,,,
object,float64,float64,float32,float32,int16,str1,object,object,object,float64,int16,int16,object,float64,int16,float64,int16,str1,object,object,float32,str1,object,int16,float64,str1,int16,float64,int16,str1
* omi02 Eri B,63.84081549227875,-7.658112218552501,0.0368,0.0308,90,O,2020yCat.1350....0G,2MASS J04152196-0739254|GJ 166 B|Gaia DR3 3195919254111315712|* omi02 Eri B|* 40 Eri B|2E 957|8pc 198.24B|AAVSO 0410-07C|ADS 3093 B|BD-07 781|CNS5 1053|CCDM J04153-0739B|CSI-07 781 1|Ci 20 289|EGGR 33|G 160-60|GC 5140|GCRV 2441|GEN# +1.00026976B|HD 26976|LFT 339|LHS 24|LP 7-781 A|LPM 182|LTT 1908|NLTT 12868|SAO 131065|UBV 4141|UBV M 10045|UCAC2 29035414|WD 0413-07|WD 0413-077|WD 0413-074|Zkh 59|2E 0412.9-0745|WDS J04153-0739B|** STF 518B|PMSC 04107-0748B|WEB 3795|Gaia DR2 3195919254110817408,2009AJ....138.1681S,5.04,2,3,paral,--,--,--,--,,pc,1979IAUS...30...57E,10.0,D,,0,-21.0,,0,-7.004600677007478e-05,--,v
* omi02 Eri B,63.84081549227875,-7.658112218552501,0.0368,0.0308,90,O,2020yCat.1350....0G,2MASS J04152196-0739254|GJ 166 B|Gaia DR3 3195919254111315712|* omi02 Eri B|* 40 Eri B|2E 957|8pc 198.24B|AAVSO 0410-07C|ADS 3093 B|BD-07 781|CNS5 1053|CCDM J04153-0739B|CSI-07 781 1|Ci 20 289|EGGR 33|G 160-60|GC 5140|GCRV 2441|GEN# +1.00026976B|HD 26976|LFT 339|LHS 24|LP 7-781 A|LPM 182|LTT 1908|NLTT 12868|SAO 131065|UBV 4141|UBV M 10045|UCAC2 29035414|WD 0413-07|WD 0413-077|WD 0413-074|Zkh 59|2E 0412.9-0745|WDS J04153-0739B|** STF 518B|PMSC 04107-0748B|WEB 3795|Gaia DR2 3195919254110817408,2011ApJ...743..138G,5.0,0,2,ST-L,--,--,--,--,,pc,1979IAUS...30...57E,10.0,D,,0,-21.0,,0,-7.004600677007478e-05,--,v
* omi02 Eri B,63.84081549227875,-7.658112218552501,0.0368,0.0308,90,O,2020yCat.1350....0G,2MASS J04152196-0739254|GJ 166 B|Gaia DR3 3195919254111315712|* omi02 Eri B|* 40 Eri B|2E 957|8pc 198.24B|AAVSO 0410-07C|ADS 3093 B|BD-07 781|CNS5 1053|CCDM J04153-0739B|CSI-07 781 1|Ci 20 289|EGGR 33|G 160-60|GC 5140|GCRV 2441|GEN# +1.00026976B|HD 26976|LFT 339|LHS 24|LP 7-781 A|LPM 182|LTT 1908|NLTT 12868|SAO 131065|UBV 4141|UBV M 10045|UCAC2 29035414|WD 0413-07|WD 0413-077|WD 0413-074|Zkh 59|2E 0412.9-0745|WDS J04153-0739B|** STF 518B|PMSC 04107-0748B|WEB 3795|Gaia DR2 3195919254110817408,2020yCat.1350....0G,5.0077,4,1,paral,-0.0012,4,0.0012,4,,pc,1979IAUS...30...57E,10.0,D,,0,-21.0,,0,-7.004600677007478e-05,--,v


main_id,ra,dec,coo_err_maj,coo_err_min,coo_err_angle,coo_wavelength,coo_bibcode,ids,mesdistance.bibcode,mesdistance.dist,mesdistance.dist_prec,mesdistance.mespos,mesdistance.method,mesdistance.minus_err,mesdistance.minus_err_prec,mesdistance.plus_err,mesdistance.plus_err_prec,mesdistance.qual,mesdistance.unit,rvz_bibcode,rvz_err,rvz_qual,rvz_nature,rvz_err_prec,rvz_radvel,rvz_wavelength,rvz_radvel_prec,rvz_redshift,rvz_redshift_prec,rvz_type
,deg,deg,mas,mas,deg,,,,,,,,,,,,,,,,km / s,,,,km / s,,,,,
object,float64,float64,float32,float32,int16,str1,object,object,object,float64,int16,int16,object,float64,int16,float64,int16,str1,object,object,float32,str1,object,int16,float64,str1,int16,float64,int16,str1
* omi02 Eri C,63.83973332536918,-7.655748493915556,0.0504,0.0417,90,O,2020yCat.1350....0G,TIC 67772873|GJ 166 C|Gaia DR3 3195919254111314816|V* DY Eri|* omi02 Eri C|* 40 Eri C|2RE J041521-073855|2RE J0415-073|8pc 198.24C|AAVSO 0410-07D|ADS 3093 C|BD-07 781C|CNS5 1052|CCDM J04153-0739C|CSI-07 781 2|GCRV 2442|GEN# +1.00026976C|LFT 340|LHS 25|LTT 1909|NLTT 12869|RE J0415-073|RE J041520-073853|UBV 4142|Zkh 60|[RHG95] 742|2MASS J04152173-0739173|WDS J04153-0739C|** STF 518C|Gaia DR2 3195919322830293760|PMSC 04107-0748C|WISEA J041519.99-073955.6|WEB 3796|Karmn J04153-076,2018yCat.1345....0G,5.0136,4,2,paral,-0.008,4,0.008,4,,pc,1953GCRV..C......0W,5.0,E,,0,-45.0,,1,-0.00015009257894815775,--,v
* omi02 Eri C,63.83973332536918,-7.655748493915556,0.0504,0.0417,90,O,2020yCat.1350....0G,TIC 67772873|GJ 166 C|Gaia DR3 3195919254111314816|V* DY Eri|* omi02 Eri C|* 40 Eri C|2RE J041521-073855|2RE J0415-073|8pc 198.24C|AAVSO 0410-07D|ADS 3093 C|BD-07 781C|CNS5 1052|CCDM J04153-0739C|CSI-07 781 2|GCRV 2442|GEN# +1.00026976C|LFT 340|LHS 25|LTT 1909|NLTT 12869|RE J0415-073|RE J041520-073853|UBV 4142|Zkh 60|[RHG95] 742|2MASS J04152173-0739173|WDS J04153-0739C|** STF 518C|Gaia DR2 3195919322830293760|PMSC 04107-0748C|WISEA J041519.99-073955.6|WEB 3796|Karmn J04153-076,2020yCat.1350....0G,5.0137,4,1,paral,-0.0017,4,0.0017,4,,pc,1953GCRV..C......0W,5.0,E,,0,-45.0,,1,-0.00015009257894815775,--,v


63.8408 -7.65808
